# RecommendIt — Exploratory Data Analysis

Exploration of the MovieLens 1M dataset used to train the RecommendIt recommendation system.

Covers:
- Rating distribution and sparsity
- User activity analysis
- Item popularity (long-tail distribution)
- Genre statistics
- Temporal patterns

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

from src.features.feature_engineering import FeatureEngineer, GENRES

In [ ]:
fe = FeatureEngineer(data_dir='../data/ml-1m')
fe.load_data()

ratings = fe.ratings_df
users = fe.users_df
movies = fe.movies_df

print(f"Ratings:  {len(ratings):,}")
print(f"Users:    {ratings['user_id'].nunique():,}")
print(f"Movies:   {ratings['item_id'].nunique():,}")
print(f"Sparsity: {1 - len(ratings) / (ratings['user_id'].nunique() * ratings['item_id'].nunique()):.4%}")

## Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating value distribution
rating_counts = ratings['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Ratings per user (log scale)
user_counts = ratings.groupby('user_id').size()
axes[1].hist(user_counts, bins=50, color='coral', edgecolor='white', log=True)
axes[1].set_xlabel('Ratings per User')
axes[1].set_ylabel('Number of Users (log)')
axes[1].set_title(f'User Activity Distribution (median={user_counts.median():.0f})')

plt.tight_layout()
plt.show()

## Long-Tail Item Popularity

In [ ]:
item_popularity = ratings.groupby('item_id').size().sort_values(ascending=False).reset_index()
item_popularity.columns = ['item_id', 'n_ratings']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative distribution
cumulative = item_popularity['n_ratings'].cumsum() / item_popularity['n_ratings'].sum()
axes[0].plot(range(len(cumulative)), cumulative.values, color='steelblue', linewidth=2)
axes[0].axhline(0.8, color='red', linestyle='--', alpha=0.7, label='80% of ratings')
axes[0].set_xlabel('Items (sorted by popularity)')
axes[0].set_ylabel('Cumulative Fraction of Ratings')
axes[0].set_title('Long-Tail Item Popularity')
axes[0].legend()

# Log-log popularity distribution
axes[1].loglog(range(1, len(item_popularity) + 1), item_popularity['n_ratings'].values,
               color='coral', linewidth=1.5, alpha=0.8)
axes[1].set_xlabel('Item Rank (log)')
axes[1].set_ylabel('Rating Count (log)')
axes[1].set_title('Popularity Power Law')

# How much of catalog covers 80% of ratings?
idx_80 = (cumulative >= 0.80).idxmax()
pct_items = idx_80 / len(item_popularity)
print(f"Top {pct_items:.1%} of items account for 80% of all ratings")
print(f"Most popular item: {item_popularity['n_ratings'].iloc[0]:,} ratings")
print(f"Median item:       {item_popularity['n_ratings'].median():.0f} ratings")

plt.tight_layout()
plt.show()

## Genre Analysis

In [ ]:
genre_counts = {genre: 0 for genre in GENRES}
for genres_str in movies['genres']:
    for g in genres_str.split('|'):
        if g in genre_counts:
            genre_counts[g] += 1

genre_df = pd.Series(genre_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(genre_df.index, genre_df.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of Movies')
ax.set_title('MovieLens 1M — Genre Distribution')
for bar, val in zip(bars, genre_df.values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=10)
plt.tight_layout()
plt.show()

## User Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gender
gender_counts = users['gender'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values, color=['steelblue', 'coral'])
axes[0].set_title('Gender Distribution')
axes[0].set_ylabel('Users')

# Age group
age_map = {1: '<18', 18: '18-24', 25: '25-34', 35: '35-44', 45: '45-49', 50: '50-55', 56: '56+'}
users['age_label'] = users['age'].map(age_map)
age_order = ['<18', '18-24', '25-34', '35-44', '45-49', '50-55', '56+']
age_counts = users['age_label'].value_counts().reindex(age_order)
axes[1].bar(age_counts.index, age_counts.values, color='steelblue')
axes[1].set_title('Age Group Distribution')
axes[1].tick_params(axis='x', rotation=30)

# Average rating by gender
merged = ratings.merge(users[['user_id', 'gender']], on='user_id')
gender_avg = merged.groupby('gender')['rating'].mean()
axes[2].bar(gender_avg.index, gender_avg.values, color=['steelblue', 'coral'])
axes[2].set_title('Average Rating by Gender')
axes[2].set_ylabel('Mean Rating')
axes[2].set_ylim(3.0, 4.5)

plt.tight_layout()
plt.show()

## Temporal Patterns

In [ ]:
ratings_ts = ratings.copy()
ratings_ts['month'] = ratings_ts['timestamp'].dt.to_period('M')
monthly = ratings_ts.groupby('month').size().reset_index()
monthly.columns = ['month', 'count']
monthly['month_dt'] = monthly['month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['month_dt'], monthly['count'], alpha=0.3, color='steelblue')
ax.plot(monthly['month_dt'], monthly['count'], color='steelblue', linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Ratings')
ax.set_title('Rating Activity Over Time')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.show()

## Feature Engineering Preview

In [ ]:
user_features = fe.build_user_features()
item_features = fe.build_item_features()

print("User features shape:", user_features.shape)
print("Item features shape:", item_features.shape)
print("\nUser feature sample:")
display(user_features.drop(columns=['genre_pref']).head())
print("\nItem feature sample:")
display(item_features.drop(columns=['genre_vector']).head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

user_features['avg_rating'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('User Average Rating Distribution')
axes[0].set_xlabel('Average Rating')

item_features['popularity_score'].hist(bins=30, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Item Popularity Score Distribution')
axes[1].set_xlabel('Popularity Score (normalized)')

plt.tight_layout()
plt.show()

print(f"\nUser avg rating: mean={user_features['avg_rating'].mean():.2f}, std={user_features['avg_rating'].std():.2f}")
print(f"Item popularity:  mean={item_features['popularity_score'].mean():.4f}, std={item_features['popularity_score'].std():.4f}")